In [ ]:
print("Resumes with missing/empty Resume text:", resumes["Resume"].isna().sum())
print("Resumes with missing Category:", resumes["Category"].isna().sum())

In [ ]:
# Data Quality Checks on Cleaned Data
print("\n--- Data Quality Checks on Resumes (Cleaned) ---")
print("Missing values in resumes_clean:\n", resumes.isnull().sum())
print("Duplicate rows in resumes_clean:", resumes.duplicated().sum())

print("\n--- Data Quality Checks on Job Corpus (Cleaned) ---")
print("Missing values in job_corpus_clean:\n", job_corpus.isnull().sum())
print("Duplicate rows in job_corpus_clean:", job_corpus.duplicated().sum())

In [ ]:
import os

resumes["clean_text"] = resumes["Resume"].apply(clean_text)
job_corpus["clean_text"] = (
    job_corpus["title"].fillna("") + " " +
    job_corpus["description"].fillna("") + " " +
    job_corpus["skills"].fillna("")
).apply(clean_text)

print("Resumes cleaned. Sample:")
print(resumes["clean_text"].iloc[0][:150])

print("\nJob corpus cleaned. Sample:")
print(job_corpus["clean_text"].iloc[0][:150])

# Create the directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save cleaned versions
resumes.to_csv("../data/processed/resumes_clean.csv", index=False)
job_corpus.to_csv("../data/processed/job_corpus_clean.csv", index=False)
print("\nSaved cleaned datasets to data/processed/")

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Original resume snippet:", resumes["Resume"].iloc[0][:150])
print("\nCleaned:", clean_text(resumes["Resume"].iloc[0])[:150])

In [ ]:
import os

# Create the directory if it doesn't exist
os.makedirs("../data/interim", exist_ok=True)

job_corpus.to_csv("../data/interim/job_corpus.csv", index=False)
print("Saved job corpus:", job_corpus.shape)

In [ ]:
# Standardize naukri to common schema
naukri_clean = naukri.rename(columns={
    "jobtitle": "title",
    "joblocation_address": "location",
    "jobdescription": "description"
})
naukri_clean = naukri_clean[["title", "company", "location", "skills", "description", "experience"]]
naukri_clean["source"] = "naukri"

# Standardize linkedin to common schema (missing fields filled with empty string)
linkedin_clean = linkedin.copy()
linkedin_clean["company"] = ""
linkedin_clean["location"] = ""
linkedin_clean["experience"] = ""
linkedin_clean["skills"] = "" # Add missing 'skills' column
linkedin_clean = linkedin_clean[["title", "company", "location", "skills", "description", "experience"]]
linkedin_clean["source"] = "linkedin"

# Combine into one job corpus
job_corpus = pd.concat([naukri_clean, linkedin_clean], ignore_index=True)

print("Job corpus shape:", job_corpus.shape)
print("\nSource breakdown:\n", job_corpus["source"].value_counts())
print("\nSample rows:")
job_corpus.head(3)

In [ ]:
# Drop duplicate resumes
print("Before dedup:", resumes.shape)
resumes = resumes.drop_duplicates().reset_index(drop=True)
print("After dedup:", resumes.shape)

# Fill missing skills in naukri with empty string (can't drop 45% of rows)
naukri["skills"] = naukri["skills"].fillna("")

# Confirm no more missing values
print("\nNaukri missing after fix:\n", naukri.isnull().sum())

In [ ]:
# Check resume text readability
print("Sample resume text:")
print(repr(resumes["Resume"].iloc[0][:400]))

# Check for missing values in each dataset
print("\nMissing values - Resumes:\n", resumes.isnull().sum())
print("\nMissing values - Naukri:\n", naukri.isnull().sum())
print("\nMissing values - LinkedIn:\n", linkedin.isnull().sum())

# Check for duplicate rows
print("\nDuplicate resumes:", resumes.duplicated().sum())
print("Duplicate naukri postings:", naukri.duplicated().sum())
print("Duplicate linkedin postings:", linkedin.duplicated().sum())

In [ ]:
print("Number of unique categories:", resumes["Category"].nunique())
print("\nCategory counts:")
print(resumes["Category"].value_counts())

print("\nSample resume text:")
print(resumes["Resume"].iloc[0][:300])

In [ ]:
import pandas as pd

BASE = "../data/raw"

# Resume Dataset
resumes = pd.read_csv(
    f"{BASE}/resumes/resumes.csv",
    encoding="utf-8"
)

# Naukri Dataset
naukri = pd.read_csv(
    f"{BASE}/naukri/naukri.csv"
)

# LinkedIn Dataset
linkedin = pd.read_csv(
    f"{BASE}/linkedin/postings.csv"
)

# Fix encoding
def fix_encoding(text):
    try:
        return text.encode("latin-1", errors="ignore").decode("utf-8", errors="ignore")
    except:
        return text

resumes["Resume"] = resumes["Resume"].apply(fix_encoding)

print("Resumes:", resumes.shape)
print("Naukri:", naukri.shape)
print("LinkedIn:", linkedin.shape)
print("Sample fixed text:")
print(resumes["Resume"].iloc[0][:200])